In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("HDI_DATASET.xlsx", header=1)
df.columns = df.columns.str.strip()

# WE USE KAKWANI TRANSFORMATION TO CALCULATE HDI_3
def kakwani_transform(x, minimum, maximum):
    return (
        np.log(maximum - minimum)
        - np.log(maximum - x)
    ) / np.log(maximum - minimum)


def calculate_hdi3(df):

    output_df = df.copy()

    # Goalposts
    LE_MIN, LE_MAX = 20.0, 85.0
    SCH_MIN, SCH_MAX = 0.0, 15.0

    # Income goalposts used by UNDP
    INC_MIN, INC_MAX = 100.0, 75000.0

    # Kakwani convex transformation
    output_df["education_index"] = kakwani_transform(
        output_df["MEAN YEARS OF SCHOOLING"],
        SCH_MIN,
        SCH_MAX
    )

    output_df["health_index"] = kakwani_transform(
        output_df["LIFE EXPECTANCY"],
        LE_MIN,
        LE_MAX
    )

    # Logarithmic income transformation
    income = np.clip(
        output_df["GDP per Capita (Constant 2015 US$)"],
        INC_MIN,
        INC_MAX
    )

    output_df["income_index"] = (
        np.log(income) - np.log(INC_MIN)
    ) / (
        np.log(INC_MAX) - np.log(INC_MIN)
    )

    # Geometric mean
    output_df["HDI3"] = (
        output_df["education_index"]
        * output_df["health_index"]
        * output_df["income_index"]
    ) ** (1 / 3)

    return output_df


HDI_3 = calculate_hdi3(df)

HDI_3[
    ["Year", "education_index", "health_index", "income_index", "HDI3"]
].head(33)

,Year,education_index,health_index,income_index,HDI3
0,1990,0.076296,0.219503,0.254152,0.162061
1,1991,0.079336,0.219503,0.252457,0.163819
2,1992,0.082400,0.219503,0.257292,0.166955
3,1993,0.085490,0.228899,0.261148,0.172246
4,1994,0.088606,0.228899,0.267784,0.175778
5,1995,0.091749,0.238678,0.275779,0.182106
6,1996,0.098116,0.238678,0.283766,0.188005
7,1997,0.101341,0.238678,0.286782,0.190714
8,1998,0.107877,0.248873,0.292923,0.198863
9,1999,0.114531,0.248873,0.302844,0.205135


In [ ]:
# NON-INCOME HDI 3
# Geometric mean of transformed Education and Health
HDI_3["NIHDI3"] = (
    HDI_3["education_index"] * HDI_3["health_index"]
) ** (1 / 2)

# Display the result
HDI_3[["Year", "education_index", "health_index", "NIHDI3"]].head(33)


,Year,education_index,health_index,NIHDI3
0,1990,0.076296,0.219503,0.129411
1,1991,0.079336,0.219503,0.131964
2,1992,0.082400,0.219503,0.134488
3,1993,0.085490,0.228899,0.139888
4,1994,0.088606,0.228899,0.142414
5,1995,0.091749,0.238678,0.147981
6,1996,0.098116,0.238678,0.153030
7,1997,0.101341,0.238678,0.155525
8,1998,0.107877,0.248873,0.163853
9,1999,0.114531,0.248873,0.168830


In [ ]:
ts3 = HDI_3[[
    "Year",
    "HDI3",
    "NIHDI3"
]].copy()

ts3.to_csv("ts_HDI3.csv", index=False)